In [ ]:
import snowflake.connector
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import logging
import datetime

# Generate a timestamped filename
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f'load_parallel_{timestamp}.log'

# Clear existing handlers if any (important if running in interactive env)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
# Snowflake connection parameters
conn_params = {
'account' : "",
'user' : "JOHN_DOE",
'password' : "<PSWD>",
'role' : "JOHN_DOE",
'warehouse' : "COMPUTE_WH",
'database' : "SNOWFLAKE_LEARNING_DB",
'schema' :"SNOWFLAKE_TEST"
}

MIN_ROWS_PER_CHUNK = 30_000_000_000  # target ~30 billion rows per chunk

# Map warehouse size to max concurrency (approximate)
CONCURRENCY_MAP = {
    'X-SMALL': 1,
    'SMALL': 2,
    'MEDIUM': 4,
    'LARGE': 8,
    'X-LARGE': 16,
    '2X-LARGE': 32,
    '3X-LARGE': 64,
}

# def get_warehouse_size():
#     with snowflake.connector.connect(**conn_params) as conn:
#         with conn.cursor() as cur:
#             cur.execute("SELECT WAREHOUSE_SIZE FROM INFORMATION_SCHEMA.WAREHOUSES WHERE WAREHOUSE_NAME = CURRENT_WAREHOUSE()")
#             size = cur.fetchone()
#             if size:
#                 return size[0].upper()
#     return None

def get_source_row_count():
    with snowflake.connector.connect(**conn_params) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1000.ORDERS")
            return cur.fetchone()[0]

def calculate_chunks(total_rows, min_rows_per_chunk, max_concurrency):
    n = total_rows // min_rows_per_chunk
    n = max(1, min(n, max_concurrency))
    return n

def prepare_queries(n):
    queries = []
    for i in range(n):
        query = f"""
        CREATE OR REPLACE TABLE SNOWFLAKE_LEARNING_DB.SNOWFLAKE_TEST.ORDERS_{i} AS
        WITH numbered_rows AS (
            SELECT *, ROW_NUMBER() OVER (ORDER BY HASH(*)) AS rn
            FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1000.ORDERS
        )
        SELECT *
        FROM numbered_rows
        WHERE MOD(rn, {n}) = {i};
        """
        queries.append(query)
    return queries

MAX_RETRIES = 3

def run_query_with_retry(query, idx):
    attempt = 0
    while attempt < MAX_RETRIES:
        try:
            with snowflake.connector.connect(**conn_params) as conn:
                with conn.cursor() as cur:
                   # cur.execute("ALTER SESSION SET ENABLE_QUERY_ACCELERATION_SERVICE = FALSE;")
                    logging.info(f"Chunk {idx}: Starting query (attempt {attempt + 1})")
                    cur.execute(query)
                    logging.info(f"Chunk {idx}: Query finished successfully")
                    return True
        except Exception as e:
            logging.error(f"Chunk {idx}: Query failed on attempt {attempt + 1}: {e}")
            attempt += 1
            time.sleep(10)
    logging.error(f"Chunk {idx}: Max retries reached. Query failed.")
    return False

def verify_row_counts(n):
    total_source = 0
    total_target = 0
    try:
        with snowflake.connector.connect(**conn_params) as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1000.ORDERS")
                total_source = cur.fetchone()[0]
                logging.info(f"Source table row count: {total_source}")

                for i in range(n):
                    cur.execute(f"SELECT COUNT(*) FROM SNOWFLAKE_LEARNING_DB.SNOWFLAKE_TEST.ORDERS_{i}")
                    cnt = cur.fetchone()[0]
                    total_target += cnt
                    logging.info(f"SNOWFLAKE_LEARNING_DB.SNOWFLAKE_TEST.ORDERS_{i} row count: {cnt}")

        if total_source == total_target:
            logging.info("Row count verification PASSED: totals match.")
        else:
            logging.error(f"Row count verification FAILED: source({total_source}) != target total({total_target})")
    except Exception as e:
        logging.error(f"Error during row count verification: {e}")

def main():
    # warehouse_size = get_warehouse_size()
    # if warehouse_size is None:
    #     logging.warning("Warehouse size not found, defaulting max concurrency to 4")
    #     max_concurrency = 4
    # else:
    #     max_concurrency = CONCURRENCY_MAP.get(warehouse_size, 4)
    # logging.info(f"Warehouse size: {warehouse_size}, max concurrency set to: {max_concurrency}")

    total_rows = get_source_row_count()
    logging.info(f"Total rows in source table: {total_rows}")

    n = calculate_chunks(total_rows, MIN_ROWS_PER_CHUNK, 8)
    logging.info(f"Calculated number of chunks: {n}")

    queries = prepare_queries(n)

    with ThreadPoolExecutor(max_workers=n) as executor:
        future_to_idx = {executor.submit(run_query_with_retry, q, i): i for i, q in enumerate(queries)}

        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                success = future.result()
                if not success:
                    logging.error(f"Chunk {idx}: Failed after retries.")
            except Exception as e:
                logging.error(f"Chunk {idx}: Unexpected error: {e}")

    logging.info("Parallel load job finished, starting verification")
    verify_row_counts(n)
    logging.info("Verification complete")

if __name__ == "__main__":
    logging.info("Starting parallel load job")
    main()


In [ ]:
import snowflake.connector
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
import datetime
import threading
from collections import defaultdict

# ==================== LOGGING SETUP ====================
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f'load_parallel_{timestamp}.log'
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s [Thread %(threadName)s - ID %(thread)d] %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# ==================== SNOWFLAKE CONNECTION ====================
conn_params = {
    'account': "",
    'user': "JOHN_DOE",
    'password': "<PSWD>",
    'role': "JOHN_DOE",
    'warehouse': "COMPUTE_WH",
    'database': "SNOWFLAKE_LEARNING_DB",
    'schema': "SNOWFLAKE_TEST"
}

MIN_ROWS_PER_CHUNK = 30_000_000_000
MAX_CONCURRENCY = 10

# ==================== GLOBAL SUMMARY STORE ====================
thread_row_counts = defaultdict(int)

# ==================== FUNCTIONS ====================

def get_source_row_count():
    """Count rows in the source table."""
    with snowflake.connector.connect(**conn_params) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.CALL_CENTER")
            return cur.fetchone()[0]

def calculate_chunks(total_rows, min_rows_per_chunk, max_concurrency):
    """Calculate optimal number of chunks based on row count and limits."""
    if total_rows <= min_rows_per_chunk:
        # Not enough rows to split — run single-threaded
        return 1
    else:
        n = total_rows // min_rows_per_chunk
        return max(1, min(n, max_concurrency))


def prepare_queries(n):
    """Generate SQL queries for each chunk."""
    queries = []
    for i in range(n):
        query = f"""
        INSERT INTO SNOWFLAKE_LEARNING_DB.SNOWFLAKE_TEST.CALL_CENTER
        SELECT *
        FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.CALL_CENTER
        WHERE MOD(ABS(HASH(CC_MKT_ID)), {n}) = {i};
        """
        queries.append((i, query))
    return queries

def run_query_with_rowcount(chunk_info):
    """Run a query and log thread info, MOD condition, and row count inserted."""
    idx, query = chunk_info
    thread_id = threading.get_ident()
    try:
        with snowflake.connector.connect(**conn_params) as conn:
            with conn.cursor() as cur:
                cur.execute("ALTER SESSION SET USE_CACHED_RESULT = FALSE;")
                logging.info(f"Chunk {idx} (Thread ID {thread_id}): Starting — MOD(ABS(HASH(CC_MKT_ID)), total_chunks) = {idx}")
                cur.execute(query)
                rowcount = cur.rowcount
                logging.info(f"Chunk {idx} (Thread ID {thread_id}): Inserted {rowcount:,} rows successfully")

                # Save to summary
                thread_row_counts[thread_id] += rowcount

                return True
    except Exception as e:
        logging.error(f"Chunk {idx} (Thread ID {thread_id}): Query failed: {e}")
        return False

def verify_row_counts():
    """Verify row counts between source and target table."""
    try:
        with snowflake.connector.connect(**conn_params) as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.CALL_CENTER")
                total_source = cur.fetchone()[0]
                cur.execute("SELECT COUNT(*) FROM SNOWFLAKE_LEARNING_DB.SNOWFLAKE_TEST.CALL_CENTER")
                total_target = cur.fetchone()[0]
                logging.info(f"Source row count: {total_source:,}, Target row count: {total_target:,}")
                if total_source == total_target:
                    logging.info("Row count verification PASSED.")
                else:
                    logging.error("Row count verification FAILED.")
    except Exception as e:
        logging.error(f"Error during row count verification: {e}")

def log_summary():
    """Log a summary of rows processed by each thread."""
    logging.info("=" * 50)
    logging.info("FINAL ROW COUNT SUMMARY PER THREAD")
    for tid, count in thread_row_counts.items():
        logging.info(f"Thread ID {tid}: {count:,} rows inserted")
    total_inserted = sum(thread_row_counts.values())
    logging.info(f"TOTAL rows inserted across all threads: {total_inserted:,}")
    logging.info("=" * 50)

# ==================== MAIN ====================

def main():
    logging.info("Starting parallel load job")

    total_rows = get_source_row_count()
    logging.info(f"Total rows in source table: {total_rows:,}")

    n = calculate_chunks(total_rows, MIN_ROWS_PER_CHUNK, MAX_CONCURRENCY)
    logging.info(f"Calculated number of chunks: {n}")

    # TRUNCATE once before starting inserts
    with snowflake.connector.connect(**conn_params) as conn:
        with conn.cursor() as cur:
            cur.execute("TRUNCATE TABLE SNOWFLAKE_LEARNING_DB.SNOWFLAKE_TEST.CALL_CENTER")
            logging.info("Target table CALL_CENTER truncated before load.")
            
    queries = prepare_queries(n)

    with ThreadPoolExecutor(max_workers=n) as executor:
        future_to_idx = {executor.submit(run_query_with_rowcount, q): q[0] for q in queries}
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            if not future.result():
                logging.error(f"Chunk {idx} failed.")

    verify_row_counts()
    log_summary()
    logging.info("Parallel load job completed.")

if __name__ == "__main__":
    main()


In [ ]:
import snowflake.connector
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
import datetime
import threading
from collections import defaultdict

# ==================== LOGGING SETUP ====================
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f'load_parallel_{timestamp}.log'
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s [Thread %(threadName)s - ID %(thread)d] %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# ==================== SNOWFLAKE CONNECTION ====================
conn_params = {
    'account': "",
    'user': "JOHN_DOE",
    'password': "<PSWD>",
    'role': "JOHN_DOE",
    'warehouse': "SNOWFLAKE_TEST",
    'database': "SNOWFLAKE_LEARNING_DB",
    'schema': "SNOWFLAKE_TEST"
}

MIN_ROWS_PER_CHUNK = 30_000_000_000
MAX_CONCURRENCY = 10

# ==================== GLOBAL SUMMARY STORE ====================
thread_row_counts = defaultdict(int)

# ==================== FUNCTIONS ====================

def get_source_row_count():
    """Count rows in the source table."""
    with snowflake.connector.connect(**conn_params) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.STORE_SALES")
            return cur.fetchone()[0]

def calculate_chunks(total_rows, min_rows_per_chunk, max_concurrency):
    """Calculate optimal number of chunks based on row count and limits."""
    if total_rows <= min_rows_per_chunk:
        # Not enough rows to split — run single-threaded
        return 1
    else:
        n = total_rows // min_rows_per_chunk
        return max(1, min(n, max_concurrency))


def prepare_queries(n):
    """Generate SQL queries for each chunk."""
    queries = []
    for i in range(n):
        query = f"""
        INSERT INTO SNOWFLAKE_LEARNING_DB.SNOWFLAKE_TEST.STORE_SALES
        SELECT *
        FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.STORE_SALES
        WHERE MOD(ABS(HASH(SS_SOLD_DATE_SK, SS_TICKET_NUMBER)), {n}) = {i};
        """
        queries.append((i, query))
    return queries

def run_query_with_rowcount(chunk_info):
    """Run a query and log thread info, MOD condition, and row count inserted."""
    idx, query = chunk_info
    thread_id = threading.get_ident()
    try:
        with snowflake.connector.connect(**conn_params) as conn:
            with conn.cursor() as cur:
                cur.execute("ALTER SESSION SET USE_CACHED_RESULT = FALSE;")
                logging.info(f"Chunk {idx} (Thread ID {thread_id}): Starting — MOD(ABS(HASH(SS_SOLD_DATE_SK, SS_TICKET_NUMBER)), total_chunks) = {idx}")
                cur.execute(query)
                rowcount = cur.rowcount
                logging.info(f"Chunk {idx} (Thread ID {thread_id}): Inserted {rowcount:,} rows successfully")

                # Save to summary
                thread_row_counts[thread_id] += rowcount

                return True
    except Exception as e:
        logging.error(f"Chunk {idx} (Thread ID {thread_id}): Query failed: {e}")
        return False

def verify_row_counts():
    """Verify row counts between source and target table."""
    try:
        with snowflake.connector.connect(**conn_params) as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.STORE_SALES")
                total_source = cur.fetchone()[0]
                cur.execute("SELECT COUNT(*) FROM SNOWFLAKE_LEARNING_DB.SNOWFLAKE_TEST.STORE_SALES")
                total_target = cur.fetchone()[0]
                logging.info(f"Source row count: {total_source:,}, Target row count: {total_target:,}")
                if total_source == total_target:
                    logging.info("Row count verification PASSED.")
                else:
                    logging.error("Row count verification FAILED.")
    except Exception as e:
        logging.error(f"Error during row count verification: {e}")

def log_summary():
    """Log a summary of rows processed by each thread."""
    logging.info("=" * 50)
    logging.info("FINAL ROW COUNT SUMMARY PER THREAD")
    for tid, count in thread_row_counts.items():
        logging.info(f"Thread ID {tid}: {count:,} rows inserted")
    total_inserted = sum(thread_row_counts.values())
    logging.info(f"TOTAL rows inserted across all threads: {total_inserted:,}")
    logging.info("=" * 50)

# ==================== MAIN ====================

def main():
    logging.info("Starting parallel load job")

    total_rows = get_source_row_count()
    logging.info(f"Total rows in source table: {total_rows:,}")

    n = calculate_chunks(total_rows, MIN_ROWS_PER_CHUNK, MAX_CONCURRENCY)
    logging.info(f"Calculated number of chunks: {n}")

    # TRUNCATE once before starting inserts
    with snowflake.connector.connect(**conn_params) as conn:
        with conn.cursor() as cur:
            cur.execute("TRUNCATE TABLE SNOWFLAKE_LEARNING_DB.SNOWFLAKE_TEST.STORE_SALES")
            logging.info("Target table STORE_SALES truncated before load.")

    queries = prepare_queries(n)

    with ThreadPoolExecutor(max_workers=n) as executor:
        future_to_idx = {executor.submit(run_query_with_rowcount, q): q[0] for q in queries}
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            if not future.result():
                logging.error(f"Chunk {idx} failed.")

    verify_row_counts()
    log_summary()
    logging.info("Parallel load job completed.")

if __name__ == "__main__":
    main()


In [ ]:
#https://www.geeksforgeeks.org/python/using-python-environment-variables-with-python-dotenv/
from dotenv import load_dotenv
import os

In [ ]:
# Access environment variables as if they came from the actual environment
load_dotenv()
SECRET_KEY = os.getenv('DOMAIN')
print(f'SECRET_KEY: {SECRET_KEY}')

In [ ]:
import sys
PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)